In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ks.disable
    ld.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

disable_all_ps()

# Test A7

kt : ---> VDD3V3

ps.ch1 : ---> relay power

ks ---> dm.ch1 current input (I) ---> relay.ch1

ks +--> dm.ch2 current input (I) ---> relay.ch2

dm.ch1 : ---> relay.ch1 (I_VIN)

dm.ch2 : ---> relay.ch2 (I_VOUT)

relay.ch1 : VIN

relay.ch2 : VOUT

loader : shall be disconnected

In [ ]:
temperature = "N40C"

log.output_set_filename(f"[{temperature}_7] I_LK_VIN_OFF, R_VIN_PD")

In [ ]:
disable_all_ps()
relay.init_channels

v_vdd = 3.3
v_en = 0

v_start  = 5
v_finish = 30.1
step = 1
ndigit = 1

list_temp = list(np.arange(v_start, v_finish, step))
list_vin_vout_lk = [round(num, ndigit) for num in list_temp]

print(f"voltage step : {step}, round : {ndigit}")

# --------------------------------------------
ps.ch1.cfg_all = 5, 1 # relay
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

ks.cfg_all = list_vin_vout_lk[0], 0.5 # vin
ks.enable
delay(0.5)

dm.ch1.current_2mA
dm.ch2.current_2mA

ic.VIN_DIS_R_EN = 0
ic.VOUT_DIS_R_EN = 0

In [ ]:
# ----------------------------------------------------------------------------
# VIN first
# ----------------------------------------------------------------------------

relay.ch1.enable
relay.ch2.disable
delay(1)

log.output_csv(["VIN (V)", "VOUT (V)", "VDD3V3 (V)", "EN (V)", "I_VIN (uA, PD_OFF)"])

for voltage in list_vin_vout_lk:
    
    ks.vset = voltage
    
    ic.VIN_DIS_R_EN = 0
    i_vin_pd_off = round(dm.ch1.current_2mA * 1e+6, 6) # uA unit
    
    ret = [voltage, 0, v_vdd, 0, i_vin_pd_off]
    log.output_csv(ret)
    print(ret)

# ----------------------------------------------------------------------------

ks.vset = list_vin_vout_lk[0]
delay(1)

log.output_csv(["VIN (V)", "VOUT (V)", "VDD3V3 (V)", "EN (V)", "I_VIN (mA, PD_ON)"])

for voltage in list_vin_vout_lk:
    
    ks.vset = voltage
    
    ic.VIN_DIS_R_EN = 1
    i_vin_pd_on = round(dm.ch1.current_20mA * 1e+3, 6) # mA unit
    
    ret = [voltage, 0, v_vdd, 0, i_vin_pd_on]
    log.output_csv(ret)
    print(ret)
    
# ----------------------------------------------------------------------------

relay.init_channels

ks.vset = list_vin_vout_lk[0]
delay(1)

ic.VIN_DIS_R_EN = 0
ic.VOUT_DIS_R_EN = 0

# ----------------------------------------------------------------------------
# VOUT Second
# ----------------------------------------------------------------------------

relay.ch1.disable
relay.ch2.enable
delay(1)

log.output_csv(["VIN (V)", "VOUT (V)", "VDD3V3 (V)", "EN (V)", "I_VOUT (uA, PD_OFF)"])

for voltage in list_vin_vout_lk:
    
    ks.vset = voltage
    
    ic.VOUT_DIS_R_EN = 0
    i_vout_pd_off = round(dm.ch2.current_2mA * 1e+6, 6) # uA unit
    
    ret = [0, voltage, v_vdd, 0, i_vout_pd_off]
    log.output_csv(ret)
    print(ret)


ks.vset = list_vin_vout_lk[0]
delay(1)

log.output_csv(["VIN (V)", "VOUT (V)", "VDD3V3 (V)", "EN (V)", "I_VOUT (mA, PD_ON)"])

for voltage in list_vin_vout_lk:
    
    ks.vset = voltage
    
    ic.VOUT_DIS_R_EN = 1
    i_vout_pd_on = round(dm.ch2.current_20mA * 1e+3, 6) # mA unit
    
    ret = [0, voltage, v_vdd, 0, i_vout_pd_on]
    log.output_csv(ret)
    print(ret)
    
# ----------------------------------------------------------------------------

relay.init_channels

ks.vset = list_vin_vout_lk[0]
delay(1)

# ----------------------------------------------------------------------------
# VIN & VOUT, PD OFF
# ----------------------------------------------------------------------------

relay.ch1.enable
relay.ch2.enable
delay(1)

ic.VIN_DIS_R_EN = 0
ic.VOUT_DIS_R_EN = 0

log.output_csv(["VIN (V)", "VOUT (V)", "VDD3V3 (V)", "EN (V)", "I_VIN (uA, PD_OFF)", "I_VOUT (uA, PD_OFF)"])

for voltage in list_vin_vout_lk:
    
    ks.vset = voltage
    
    ic.VIN_DIS_R_EN = 0
    i_vin_pd_off = round(dm.ch1.current_2mA * 1e+6, 6) # uA unit
    
    ic.VOUT_DIS_R_EN = 0
    i_vout_pd_off = round(dm.ch2.current_2mA * 1e+6, 6) # uA unit
    
    ret = [voltage, voltage, v_vdd, 0, i_vin_pd_off, i_vout_pd_off]
    log.output_csv(ret)
    print(ret)
    
# ----------------------------------------------------------------------------

ks.vset = list_vin_vout_lk[0]
delay(1)

# ----------------------------------------------------------------------------
# VIN & VOUT, PD ON
# ----------------------------------------------------------------------------

log.output_csv(["VIN (V)", "VOUT (V)", "VDD3V3 (V)", "EN (V)", "I_VIN (mA, PD_ON)", "I_VOUT (mA, PD_ON)"])

for voltage in list_vin_vout_lk:
    
    ks.vset = voltage
    
    ic.VIN_DIS_R_EN = 1
    i_vin_pd_on = round(dm.ch1.current_20mA * 1e+3, 6) # mA unit
    
    ic.VOUT_DIS_R_EN = 1
    i_vout_pd_on = round(dm.ch2.current_20mA * 1e+3, 6) # mA unit
    
    ret = [voltage, voltage, v_vdd, 0, i_vin_pd_on, i_vout_pd_on]
    log.output_csv(ret)
    print(ret)
    
# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()

ks.cfg_all = 0.1, 0.5